In [4]:
import polars as pl
import zipfile

# 1. Define your local file path and Binance trade schema
file_path = "BTCUSDT-trades-2026-06-22.zip"
columns = [
    "trade_id", "price", "qty", "quote_qty", 
    "timestamp", "is_buyer_maker", "is_best_match"
]

# 2. Extract and read the CSV directly from the local ZIP
with zipfile.ZipFile(file_path) as z:
    # Dynamically find the .csv file name inside the zip archive
    csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
    
    with z.open(csv_filename) as csv_file:
        df = pl.read_csv(csv_file.read(), has_header=False, new_columns=columns)

# 3. Clean up the data types and format the timestamp
df_clean = df.with_columns([
    pl.from_epoch("timestamp", time_unit="ms"),
    pl.col(["price", "qty", "quote_qty"]).cast(pl.Float64)
])
df_clean = df.with_columns([
    # Divide the raw timestamp to get true seconds, then parse
    (pl.col("timestamp") / 1000).cast(pl.Datetime("ms")),
    pl.col(["price", "qty", "quote_qty"]).cast(pl.Float64)
])

# 4. Display the results in your notebook
print(f"Successfully loaded {df_clean.shape[0]:,} trade records.")
df_clean.head()


Successfully loaded 3,521,337 trade records.


trade_id,price,qty,quote_qty,timestamp,is_buyer_maker,is_best_match
i64,f64,f64,f64,datetime[ms],bool,bool
6428978794,63312.0,0.00036,22.79232,2026-06-22 00:00:00.155,false,true
6428978795,63312.0,0.00007,4.43184,2026-06-22 00:00:00.191,false,true
6428978796,63312.0,0.00548,346.94976,2026-06-22 00:00:00.195,false,true
6428978797,63312.0,0.0012,75.9744,2026-06-22 00:00:00.195,false,true
6428978798,63312.0,0.008,506.496,2026-06-22 00:00:00.195,false,true


In [5]:
len(df_clean)

3521337